# xTB features: ion-pair interaction energy and solvation free energy
Jilla Sai Mahima · DDP · Guide: Prof. Sanjib Senapati

Two physics features for the counter-ion recommender, computed with **GFN2-xTB** (fast semi-empirical quantum chemistry) in implicit water (ALPB):

| Feature | What it is | Level |
|---|---|---|
| **Solvation free energy (Gsolv)** | How strongly water holds the molecule (gas → water), kcal/mol. More negative = more hydrophilic. | every ion and every drug |
| **Ion-pair interaction energy (ΔE_int)** | E(drug ion + counter-ion together) − E(drug ion) − E(counter-ion), kcal/mol. More negative = binds more strongly. | per drug–ion pair |

**Method**
1. Each molecule (charged form, as in our tables): RDKit conformers → lowest MMFF conformer → GFN2-xTB optimisation in water → Gsolv.
2. Each pair: the counter-ion is placed next to the drug's charged atom (charged atom about 3.2 Å from charged atom) in several random orientations. Each orientation is optimised with GFN-FF (fast force field) in water, then the energy is taken with GFN2-xTB in water. The lowest ΔE_int over orientations is kept. The two separate ions get the same GFN-FF → GFN2 treatment, so the energies are consistent.
3. Which pairs: all known pairs + the same number of **popularity-matched non-pairs** (for every true ion of a drug, an unused ion of the same charge that is about as popular). This way the test below asks "does the feature tell true pairs apart from equally common ions?", not "is the ion common?".

**Test (section 6):** for each drug, AUC = chance that a true pair scores better than its non-pairs (0.5 = no signal). Averaged over drugs with a bootstrap 95% CI.

**Run time on Colab (2 CPUs):** molecules about 15–30 min, pairs about 1–2 h. Everything is saved as it goes, so if Colab disconnects, just run all cells again and it continues where it stopped. Set `MAX_DRUGS = 10` for a quick trial first.

## 0. Install xTB and RDKit

In [ ]:
!pip -q install rdkit pandas matplotlib
import os, subprocess
if not os.path.exists('xtb-dist/bin/xtb'):
    !wget -q https://github.com/grimme-lab/xtb/releases/download/v6.7.1/xtb-6.7.1-linux-x86_64.tar.xz -O xtb.tar.xz && tar -xf xtb.tar.xz
os.environ['XTBHOME'] = os.path.abspath('xtb-dist'); os.environ['PATH'] = os.path.abspath('xtb-dist/bin') + ':' + os.environ['PATH']
print(subprocess.run(['xtb', '--version'], capture_output=True, text=True).stdout.strip().split('\n')[-3])

In [ ]:
# ------------------------- settings -------------------------
PAIRS_FILE   = 'Pairs_652_with_dpKa_reliability_and_safety.csv'
N_ORIENT     = 5          # orientations tried per pair (lowest energy kept)
NEG_PER_POS  = 1          # popularity-matched non-pairs per true pair
MAX_DRUGS    = None       # e.g. 10 for a quick trial; None = all drugs
N_WORKERS    = os.cpu_count()
SEED         = 0
# ------------------------------------------------------------

## 1. Helper functions (xTB runner, conformers, pair builder)

In [ ]:
import os, re, json, time, shutil, tempfile, subprocess
import numpy as np, pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
RDLogger.DisableLog('rdApp.*')
HARTREE = 627.5095

def run_xtb(xyz, chrg, args, keep_geom=True, timeout=1800):
    """run xtb in a temp folder; returns (total energy Eh, Gsolv Eh, optimised xyz text or None, ok)"""
    d = tempfile.mkdtemp(prefix='xtb_')
    try:
        open(os.path.join(d, 'in.xyz'), 'w').write(xyz)
        cmd = f'ulimit -s unlimited 2>/dev/null; xtb in.xyz --chrg {int(chrg)} {args} > out.txt 2>&1'
        env = dict(os.environ, OMP_NUM_THREADS='1', MKL_NUM_THREADS='1', OMP_STACKSIZE='1G')
        subprocess.run(['bash', '-c', cmd], cwd=d, env=env, timeout=timeout)
        out = open(os.path.join(d, 'out.txt')).read()
        ok = 'normal termination of xtb' in out
        e = re.findall(r'TOTAL ENERGY\s+(-?\d+\.\d+)\s+Eh', out); g = re.findall(r'-> Gsolv\s+(-?\d+\.\d+)\s+Eh', out)
        geo = None
        if keep_geom:
            for f in ('xtbopt.xyz', 'in.xtbopt.xyz'):
                p = os.path.join(d, f)
                if os.path.exists(p): geo = open(p).read(); break
        return (float(e[-1]) if e else np.nan, float(g[-1]) if g else np.nan, geo, ok and bool(e))
    except Exception:
        return np.nan, np.nan, None, False
    finally:
        shutil.rmtree(d, ignore_errors=True)

def xyz_to_arrays(xyz):
    L = xyz.strip().split('\n'); n = int(L[0]); el = []; X = []
    for l in L[2:2 + n]:
        p = l.split(); el.append(p[0]); X.append([float(v) for v in p[1:4]])
    return el, np.array(X)
def arrays_to_xyz(el, X, title=''):
    return f'{len(el)}\n{title}\n' + '\n'.join(f'{e} {x:.6f} {y:.6f} {z:.6f}' for e, (x, y, z) in zip(el, X)) + '\n'

def embed_xyz(smiles, n_conf=5, seed=42):
    """RDKit conformers -> MMFF (UFF fallback) -> lowest-energy conformer as xyz. Returns (xyz, index of charged atom in xyz)."""
    m = Chem.AddHs(Chem.MolFromSmiles(smiles))
    ps = AllChem.ETKDGv3(); ps.randomSeed = seed; ps.maxIterations = 200
    ids = list(AllChem.EmbedMultipleConfs(m, n_conf, ps))
    if not ids:
        ps.useRandomCoords = True; ids = list(AllChem.EmbedMultipleConfs(m, n_conf, ps))
    if not ids: return None
    try:
        res = AllChem.MMFFOptimizeMoleculeConfs(m, maxIters=500)
        if not res or all(r[0] == -1 for r in res): raise ValueError
    except Exception:
        res = AllChem.UFFOptimizeMoleculeConfs(m, maxIters=500)
    best = ids[int(np.argmin([r[1] for r in res]))]
    return Chem.MolToXYZBlock(m, confId=best)

def charged_atoms(smiles, sign):
    """indices (in AddHs order = xyz order) of atoms carrying the net charge sign"""
    m = Chem.AddHs(Chem.MolFromSmiles(smiles))
    idx = [a.GetIdx() for a in m.GetAtoms() if np.sign(a.GetFormalCharge()) == sign]
    return idx or [a.GetIdx() for a in m.GetAtoms() if a.GetFormalCharge() != 0] or [0]

def rand_rot(rng):
    q = rng.normal(size=4); q /= np.linalg.norm(q); a, b, c, d = q
    return np.array([[a*a+b*b-c*c-d*d, 2*(b*c-a*d), 2*(b*d+a*c)], [2*(b*c+a*d), a*a-b*b+c*c-d*d, 2*(c*d-a*b)], [2*(b*d-a*c), 2*(c*d+a*b), a*a-b*b-c*c+d*d]])
def align(u, v):
    """rotation matrix taking unit vector u to unit vector v"""
    u = u / np.linalg.norm(u); v = v / np.linalg.norm(v); w = np.cross(u, v); s = np.linalg.norm(w); c = u @ v
    if s < 1e-8: return np.eye(3) if c > 0 else -np.eye(3)
    K = np.array([[0, -w[2], w[1]], [w[2], 0, -w[0]], [-w[1], w[0], 0]]); return np.eye(3) + K + K @ K * ((1 - c) / s**2)

def build_pair(xyzA, qA, xyzB, qB, rng, dist=3.2, tries=60):
    """place B so its charged atom sits ~dist from A's charged atom, body pointing away from A. Returns xyz text."""
    elA, A = xyz_to_arrays(xyzA); elB, B = xyz_to_arrays(xyzB)
    a0 = A[rng.choice(qA)]; out_dir = a0 - A.mean(0)
    out_dir = out_dir / np.linalg.norm(out_dir) if np.linalg.norm(out_dir) > 0.3 else rng.normal(size=3)
    heavyA = np.array([e != 'H' for e in elA]); heavyB = np.array([e != 'H' for e in elB])
    for t in range(tries):
        d = out_dir + 0.7 * rng.normal(size=3); d /= np.linalg.norm(d)
        b0 = B[rng.choice(qB)]; Bc = B - b0
        body = Bc.mean(0); R1 = align(body, d) if np.linalg.norm(body) > 0.3 else rand_rot(rng)
        ang = rng.uniform(0, 2 * np.pi); k = d; K = np.array([[0, -k[2], k[1]], [k[2], 0, -k[0]], [-k[1], k[0], 0]])
        R2 = np.eye(3) + np.sin(ang) * K + (1 - np.cos(ang)) * K @ K
        Bn = (R2 @ (R1 @ Bc.T)).T + a0 + d * (dist + 0.05 * t)
        D = np.linalg.norm(A[:, None] - Bn[None], axis=2)
        if D.min() > 1.6 and D[np.ix_(heavyA, heavyB)].min() > 2.5:
            return arrays_to_xyz(elA + elB, np.vstack([A, Bn]), 'pair')
    return None

## 2. Load our drugs and counter-ions
Upload `Pairs_652_with_dpKa_reliability_and_safety.csv` (or `notebook_inputs.zip`). Same unique-ion rule as the recommender.

In [ ]:
import zipfile, warnings; warnings.filterwarnings('ignore')
from concurrent.futures import ProcessPoolExecutor, as_completed
import matplotlib.pyplot as plt
pd.set_option('display.width', 220); pd.set_option('display.max_rows', 100)
if not os.path.exists(PAIRS_FILE):
    try:
        from google.colab import files
        for f in list(files.upload()):
            if f.endswith('.zip'): zipfile.ZipFile(f).extractall('.')
    except ImportError: pass
assert os.path.exists(PAIRS_FILE), 'upload ' + PAIRS_FILE
P = pd.read_csv(PAIRS_FILE)
def frag_keys(s):
    out = set()
    for f in Chem.GetMolFrags(Chem.MolFromSmiles(s), asMols=True):
        Chem.RemoveStereochemistry(f); out.add(Chem.MolToSmiles(f))
    return tuple(sorted(out))
charge = lambda s: sum(a.GetFormalCharge() for a in Chem.MolFromSmiles(s).GetAtoms())
P['ion_key'] = P.counter_ion_smiles.map(frag_keys); assert (P.ion_key.map(len) == 1).all()
P['ion_key'] = P.ion_key.str[0]; P['drug_key'] = P.drug_smiles.map(lambda s: frag_keys(s)[0])
I = P.groupby('ion_key').agg(name=('ion_name', lambda x: x.mode().iloc[0])).reset_index().rename(columns={'ion_key': 'smiles'})
I['chg'] = I.smiles.map(charge); I['sign'] = np.sign(I.chg).astype(int); I.insert(0, 'ion_id', [f'U{i+1:03d}' for i in range(len(I))])
D = P.groupby('drug_key').agg(name=('drug', lambda x: x.mode().iloc[0]), chg_tab=('drug_chg', 'first')).reset_index().rename(columns={'drug_key': 'smiles'})
D['chg'] = D.smiles.map(charge); D['sign'] = np.where(D.chg != 0, np.sign(D.chg), np.sign(D.chg_tab.fillna(0))).astype(int)
D.insert(0, 'drug_id', [f'D{i+1:03d}' for i in range(len(D))])
di = dict(zip(D.smiles, D.drug_id)); ii = dict(zip(I.smiles, I.ion_id))
POS = P[['drug_key', 'ion_key']].drop_duplicates()
POS = pd.DataFrame({'drug_id': POS.drug_key.map(di), 'ion_id': POS.ion_key.map(ii)})
sgn_d = dict(zip(D.drug_id, D.sign)); sgn_i = dict(zip(I.ion_id, I.sign))
POS = POS[[sgn_i[j] == -sgn_d[d] for d, j in zip(POS.drug_id, POS.ion_id)]].reset_index(drop=True)
neutral_drugs = D[D.chg == 0]
print(f"{len(D)} drug forms | {len(I)} counter-ions | {len(POS)} true (drug, ion) pairs")
if len(neutral_drugs): print('drugs written without a charge (skipped for xTB):', neutral_drugs.name.tolist())

## 3. Choose the pairs to compute: true pairs + popularity-matched non-pairs

In [ ]:
rng = np.random.default_rng(SEED)
drugs_use = [d for d in D.drug_id if d in set(POS.drug_id) and D.set_index('drug_id').chg[d] != 0]
if MAX_DRUGS: drugs_use = drugs_use[:MAX_DRUGS]
users = POS.groupby('ion_id').drug_id.apply(set).to_dict()
pos_of = POS.groupby('drug_id').ion_id.apply(set).to_dict()
rows = []
for d in drugs_use:
    cand = [j for j in I.ion_id if sgn_i[j] == -sgn_d[d] and j not in pos_of[d]]
    pop = {j: len(users.get(j, set()) - {d}) for j in I.ion_id}        # how many OTHER drugs use the ion
    taken = set()
    for j in sorted(pos_of[d]):
        rows.append(dict(drug_id=d, ion_id=j, true_pair=1, matched_to=''))
        for _ in range(NEG_PER_POS):
            free = [k for k in cand if k not in taken]
            if not free: break
            gap = np.array([abs(pop[k] - pop[j]) for k in free]); best = [k for k, g in zip(free, gap) if g == gap.min()]
            k = best[rng.integers(len(best))]; taken.add(k); rows.append(dict(drug_id=d, ion_id=k, true_pair=0, matched_to=j))
PAIRS = pd.DataFrame(rows)
print(f"{len(drugs_use)} drugs | {int(PAIRS.true_pair.sum())} true pairs + {int((PAIRS.true_pair == 0).sum())} matched non-pairs = {len(PAIRS)} pairs to compute")

## 4. Every molecule: GFN2-xTB in water → Gsolv (+ reference energy for the pair step)
Saved to `xtb_molecules.csv` (and geometries to `xtb_geoms.json`) as it runs.

In [ ]:
MOLS = pd.concat([D[D.drug_id.isin(drugs_use)][['drug_id', 'smiles', 'chg', 'sign']].rename(columns={'drug_id': 'mol_id'}),
                  I[I.ion_id.isin(PAIRS.ion_id)][['ion_id', 'smiles', 'chg', 'sign']].rename(columns={'ion_id': 'mol_id'})], ignore_index=True)
MCSV, GJSON = 'xtb_molecules.csv', 'xtb_geoms.json'
def mol_job(mol_id, smiles, chg):
    t0 = time.time(); out = dict(mol_id=mol_id, smiles=smiles, chg=chg)
    xyz = embed_xyz(smiles)
    if xyz is None: return out | dict(ok=False, note='embedding failed'), None
    E, G, geo, ok = run_xtb(xyz, chg, '--opt --alpb water')
    if not ok or geo is None: return out | dict(ok=False, note='GFN2 opt failed'), None
    _, _, ff, ok2 = run_xtb(geo, chg, '--gfnff --opt --alpb water')
    Eref, _, _, ok3 = run_xtb(ff, chg, '--sp --alpb water', keep_geom=False) if ok2 and ff else (np.nan, np.nan, None, False)
    return out | dict(ok=True, E_gfn2_water=E, Gsolv_kcal=G * HARTREE, E_ref=Eref, ref_ok=ok3, sec=round(time.time() - t0, 1), note=''), {'gfn2': geo, 'ff': ff}
done = pd.read_csv(MCSV) if os.path.exists(MCSV) else pd.DataFrame(columns=['mol_id'])
geoms = json.load(open(GJSON)) if os.path.exists(GJSON) else {}
todo = MOLS[~MOLS.mol_id.isin(done.mol_id) | ~MOLS.mol_id.isin(list(geoms))]
print(len(MOLS), 'molecules,', len(todo), 'to compute'); t0 = time.time(); new = []
with ProcessPoolExecutor(N_WORKERS) as ex:
    futs = [ex.submit(mol_job, r.mol_id, r.smiles, r.chg) for r in todo.itertuples()]
    for n, f in enumerate(as_completed(futs), 1):
        row, g = f.result(); new.append(row)
        if g: geoms[row['mol_id']] = g
        if n % 25 == 0 or n == len(futs):
            done = pd.concat([done[~done.mol_id.isin([r['mol_id'] for r in new])], pd.DataFrame(new)], ignore_index=True); done.to_csv(MCSV, index=False); json.dump(geoms, open(GJSON, 'w')); new = []
            print(f'  {n}/{len(futs)}  {time.time()-t0:.0f}s', flush=True)
MOLRES = pd.read_csv(MCSV).drop_duplicates('mol_id', keep='last').set_index('mol_id')
print('ok:', int(MOLRES.ok.sum()), 'of', len(MOLRES), '| failed:', MOLRES[~MOLRES.ok.astype(bool)].index.tolist())

## 5. Every pair: ion-pair interaction energy in water
Saved to `xtb_pairs.csv` as it runs; the lowest-energy pair geometry for each pair goes to `xtb_pair_geoms.jsonl`.

In [ ]:
PCSV, PGEO = 'xtb_pairs.csv', 'xtb_pair_geoms.jsonl'
qatoms = {r.mol_id: charged_atoms(r.smiles, r.sign) for r in MOLS.itertuples()}
def pair_job(d, j, ff_d, ff_j, q_d, q_j, ch_d, ch_j, E_d, E_j, n_orient, seed):
    rng = np.random.default_rng(seed); best = (np.inf, None); es = []
    for o in range(n_orient):
        x = build_pair(ff_d, q_d, ff_j, q_j, rng)
        if x is None: continue
        _, _, g, ok = run_xtb(x, ch_d + ch_j, '--gfnff --opt --alpb water')
        if not ok or g is None: continue
        E, _, _, ok2 = run_xtb(g, ch_d + ch_j, '--sp --alpb water', keep_geom=False)
        if not ok2: continue
        dE = (E - E_d - E_j) * HARTREE; es.append(dE)
        if dE < best[0]: best = (dE, g)
    return dict(drug_id=d, ion_id=j, dE_int_kcal=(best[0] if es else np.nan), dE_mean_kcal=(np.mean(es) if es else np.nan), n_ok=len(es)), best[1]
done = pd.read_csv(PCSV) if os.path.exists(PCSV) else pd.DataFrame(columns=['drug_id', 'ion_id'])
key = set(zip(done.drug_id, done.ion_id)); okm = set(MOLRES.index[MOLRES.ok.astype(bool) & MOLRES.ref_ok.astype(bool)])
todo = [r for r in PAIRS.itertuples() if (r.drug_id, r.ion_id) not in key and r.drug_id in okm and r.ion_id in okm]
print(len(PAIRS), 'pairs,', len(todo), 'to compute'); t0 = time.time(); new = []
with ProcessPoolExecutor(N_WORKERS) as ex:
    futs = [ex.submit(pair_job, r.drug_id, r.ion_id, geoms[r.drug_id]['ff'], geoms[r.ion_id]['ff'], qatoms[r.drug_id], qatoms[r.ion_id],
                      int(MOLRES.chg[r.drug_id]), int(MOLRES.chg[r.ion_id]), MOLRES.E_ref[r.drug_id], MOLRES.E_ref[r.ion_id], N_ORIENT, SEED + k) for k, r in enumerate(todo)]
    for n, f in enumerate(as_completed(futs), 1):
        row, g = f.result(); new.append(row)
        if g:
            with open(PGEO, 'a') as fh: fh.write(json.dumps({'drug_id': row['drug_id'], 'ion_id': row['ion_id'], 'xyz': g}) + '\n')
        if n % 25 == 0 or n == len(futs):
            pd.concat([pd.read_csv(PCSV) if os.path.exists(PCSV) else None, pd.DataFrame(new)], ignore_index=True).to_csv(PCSV, index=False); new = []
            el = time.time() - t0; print(f'  {n}/{len(futs)}  {el:.0f}s  (about {el/n*(len(futs)-n)/60:.0f} min left)', flush=True)
PR = pd.read_csv(PCSV).drop_duplicates(['drug_id', 'ion_id'], keep='last')
print('pairs with an energy:', int(PR.dE_int_kcal.notna().sum()), 'of', len(PAIRS))

## 6. Do the features tell true pairs apart from equally popular non-pairs?
For each drug: AUC = chance that a true pair has the higher feature score than a non-pair (ties count half). 0.5 = no signal, above 0.5 = true pairs score higher, below 0.5 = true pairs score lower (that direction is also a signal).
Features tested:
- **stronger binding** = −ΔE_int
- **ion hydrophilicity** = −Gsolv(ion)
- **drug–ion Gsolv gap** = |Gsolv(drug) − Gsolv(ion)|
- **popularity** (number of other drugs using the ion) as a check. Should be close to 0.5 because the non-pairs were matched on it.

In [ ]:
T = PAIRS.merge(PR, on=['drug_id', 'ion_id'], how='left')
T['Gsolv_drug'] = T.drug_id.map(MOLRES.Gsolv_kcal); T['Gsolv_ion'] = T.ion_id.map(MOLRES.Gsolv_kcal)
T['pop'] = [len(users.get(j, set()) - {d}) for d, j in zip(T.drug_id, T.ion_id)]
FEATS = {'stronger binding (-dE_int)': -T.dE_int_kcal, 'ion hydrophilicity (-Gsolv ion)': -T.Gsolv_ion,
         'drug-ion Gsolv gap |dG|': (T.Gsolv_drug - T.Gsolv_ion).abs(), 'popularity (check)': T['pop'].astype(float)}
def drug_auc(v):
    out = {}
    for d, g in T.assign(v=v).dropna(subset=['v']).groupby('drug_id'):
        a, b = g.v[g.true_pair == 1].values, g.v[g.true_pair == 0].values
        if len(a) and len(b): out[d] = ((a[:, None] > b[None]).sum() + 0.5 * (a[:, None] == b[None]).sum()) / (len(a) * len(b))
    return pd.Series(out)
rb = np.random.default_rng(1); res = []
for nm, v in FEATS.items():
    s = drug_auc(v); bs = [s.values[rb.integers(0, len(s), len(s))].mean() for _ in range(2000)]
    lo, hi = np.percentile(bs, [2.5, 97.5])
    res.append(dict(feature=nm, drugs=len(s), mean_AUC=round(s.mean(), 3), CI_low=round(lo, 3), CI_high=round(hi, 3), signal=('yes' if lo > 0.5 or hi < 0.5 else 'no (CI includes 0.5)')))
AUC = pd.DataFrame(res); print(AUC.to_string(index=False))
g = T.groupby('true_pair')[['dE_int_kcal', 'Gsolv_ion']].median().rename(index={0: 'non-pair', 1: 'true pair'}); print('\nmedians (kcal/mol):'); print(g.round(2))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a, col, lab in [(ax[0], 'dE_int_kcal', 'Ion-pair interaction energy (kcal/mol)'), (ax[1], 'Gsolv_ion', 'Counter-ion Gsolv (kcal/mol)')]:
    data = [T[(T.true_pair == k)][col].dropna() for k in (1, 0)]
    a.boxplot(data, showfliers=False); a.set_xticks([1, 2]); a.set_xticklabels(['true pairs', 'matched non-pairs']); a.set_ylabel(lab)
plt.tight_layout(); plt.savefig('xtb_features_true_vs_nonpairs.png', dpi=150); plt.show()

## 7. Save everything

In [ ]:
os.makedirs('xtb_features', exist_ok=True)
mol = MOLRES.reset_index()[['mol_id', 'smiles', 'chg', 'Gsolv_kcal', 'E_gfn2_water', 'ok', 'note']]
mol.merge(pd.concat([D[['drug_id', 'name']].rename(columns={'drug_id': 'mol_id'}), I[['ion_id', 'name']].rename(columns={'ion_id': 'mol_id'})]), on='mol_id', how='left').to_csv('xtb_features/molecules_Gsolv.csv', index=False)
T.merge(D[['drug_id', 'name']].rename(columns={'name': 'drug'}), on='drug_id').merge(I[['ion_id', 'name']].rename(columns={'name': 'ion'}), on='ion_id').to_csv('xtb_features/pairs_dE_int.csv', index=False)
AUC.to_csv('xtb_features/feature_test_AUC.csv', index=False)
for f in ['xtb_features_true_vs_nonpairs.png', PGEO, GJSON]:
    if os.path.exists(f): shutil.copy(f, 'xtb_features/')
shutil.make_archive('xtb_features', 'zip', 'xtb_features')
try:
    from google.colab import files; files.download('xtb_features.zip')
except ImportError: print('saved xtb_features.zip')

## Notes and limits
- GFN2-xTB is approximate. Use the numbers to **compare** ions and pairs with each other, not as exact values.
- Pairs are 1:1 ion pairs in implicit water. Real 2:1 salts (dianions, dications) and the solid state are not modelled.
- Pair geometries come from GFN-FF, which keeps bonds fixed, so a proton cannot jump between drug and ion here. ΔpKa covers that question.
- The best of `N_ORIENT` random starting orientations is kept. More orientations = lower (better) energies but longer run time.
- Drugs written without a charge in the table are skipped (xTB needs the charged form).
- If the AUC test shows signal, add Gsolv (drug, ion, and their gap) and ΔE_int as features in the recommender and check Hit@10 with the leave-drug-out test. ΔE_int is a pair feature, so for ranking it is best used to **re-rank the top 20** suggestions per drug instead of all candidates.